In [1]:
import requests
from bs4 import BeautifulSoup
from openai import OpenAI
import gradio as gr

In [2]:
# Connect to Ollama — no API key needed
ollama = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434/v1"
)

MODEL = "llama3.2"

In [3]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

# iShopChangi is a JS-rendered SPA — requests+BeautifulSoup captures the
# static HTML shell (meta tags, SEO content, nav labels) rather than
# fully-rendered product listings.
ISHOP_PAGES = {
    "home":               "https://www.ishopchangi.com/en/home?cmode=tr",
    "liquor_tobacco":     "https://www.ishopchangi.com/en/category/liquor-and-tobacco",
    "perfumes_cosmetics": "https://www.ishopchangi.com/en/category/perfumes-and-cosmetics",
    "confectionery":      "https://www.ishopchangi.com/en/category/confectionery-and-food-beverage",
    "electronics":        "https://www.ishopchangi.com/en/category/electronics",
    "fashion":            "https://www.ishopchangi.com/en/category/fashion-watches-and-jewellery",
}

def fetch_page(url, max_chars=1500):
    try:
        resp = requests.get(url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(resp.content, "html.parser")
        title = soup.title.string.strip() if soup.title else "No title"
        if soup.body:
            for tag in soup.body(["script", "style", "img", "input", "nav", "footer"]):
                tag.decompose()
            text = soup.body.get_text(separator="\n", strip=True)
        else:
            text = ""
        return f"Page: {title}\nURL: {url}\n\n{text}"[:max_chars]
    except Exception as e:
        return f"Could not fetch {url}: {e}"

def fetch_ishop_context():
    parts = []
    for name, url in ISHOP_PAGES.items():
        print(f"Fetching {name} page...")
        parts.append(fetch_page(url))
    return "\n\n---\n\n".join(parts)

print("Loading iShopChangi content...")
ishop_context = fetch_ishop_context()
print(f"Done! Fetched {len(ishop_context)} characters of content.")

Loading iShopChangi content...
Fetching home page...
Fetching liquor_tobacco page...
Fetching perfumes_cosmetics page...
Fetching confectionery page...
Fetching electronics page...
Fetching fashion page...
Done! Fetched 606 characters of content.


In [4]:
system_message = f"""You are a friendly and knowledgeable duty-free shopping assistant for iShopChangi — Changi Airport's official online duty-free store.
Help travellers discover products, understand duty-free allowances, find promotions, and navigate the shopping categories.
You remember the conversation history so you can follow up naturally on previous messages.
If a shopper mentions a specific category (e.g. whisky, perfume, snacks), recommend exploring that section and highlight any deals.
Be concise, warm, and practical. Respond in markdown without code blocks.

Here is the current iShopChangi website content to ground your answers:

{ishop_context}
"""

In [ ]:
def chat(message, history):
    # Rebuild the message list from gradio history + new user message
    relevant_system_message = system_message
    if "iphone" in message.lower():
        relevant_system_message += (
            " If the customer asks about iPhone, let them know that iShopChangi does not sell iPhones. "
            "Politely suggest alternatives such as fragrances, premium spirits, cosmetics, confectionery, "
            "or fashion and jewellery available in the store."
        )

    messages = [{"role": "system", "content": relevant_system_message}]
    messages += [{"role": h["role"], "content": h["content"]} for h in history]
    messages.append({"role": "user", "content": message})

    stream = ollama.chat.completions.create(
        model=MODEL,
        messages=messages,
        stream=True
    )

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response

In [6]:
gr.ChatInterface(
    fn=chat,
    type="messages",
    title="iShopChangi Duty-Free Assistant",
    description="Chat with your personal duty-free shopping guide for Changi Airport. Ask about products, brands, categories, allowances, or promotions.",
    examples=[
        "What product categories are available at iShopChangi?",
        "What whisky brands can I find in the liquor section?",
        "What perfumes and cosmetics brands are available?",
        "What electronics can I buy duty-free at Changi?",
        "What local Singapore snacks can I buy as gifts?",
        "What duty-free allowances should I know about?",
    ]
).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
